# Análise Crédito Pessoal

Análise descritiva do crédito pessoal não consignado com dados públicos do Banco Central do Brasil.


In [1]:
import pandas as pd
import requests

print("Ambiente funcionando!")
print("Pandas:", pd.__version__)

Ambiente funcionando!
Pandas: 3.0.5


In [2]:
from pathlib import Path
import sys

pasta_projeto = Path.cwd().resolve()
if pasta_projeto.name == 'notebooks':
    pasta_projeto = pasta_projeto.parent
if str(pasta_projeto) not in sys.path:
    sys.path.insert(0, str(pasta_projeto))

import pandas as pd
import requests

print("Notebook funcionando!")

Notebook funcionando!


In [3]:
import sys
print(sys.executable)

c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\.venv\Scripts\python.exe


In [4]:
def buscar_serie_bcb(
    codigo_serie,
    nome_serie,
    data_inicial,
    data_final
):
    url = (
        f"https://api.bcb.gov.br/dados/serie/"
        f"bcdata.sgs.{codigo_serie}/dados"
    )

    parametros = {
        "formato": "json",
        "dataInicial": data_inicial,
        "dataFinal": data_final
    }

    resposta = requests.get(
        url,
        params=parametros,
        timeout=30
    )

    resposta.raise_for_status()
    dados = resposta.json()

    if not dados:
        raise ValueError(
            f"A série {codigo_serie} não retornou dados."
        )

    df = pd.DataFrame(dados)

    df["data"] = pd.to_datetime(
        df["data"],
        format="%d/%m/%Y"
    )

    df["valor"] = pd.to_numeric(
        df["valor"],
        errors="coerce"
    )

    df["serie"] = nome_serie
    df["codigo_serie"] = codigo_serie

    return (
        df
        .sort_values("data")
        .drop_duplicates(subset="data")
        .reset_index(drop=True)
    )

In [5]:
concessoes_teste = buscar_serie_bcb(
    codigo_serie=20666,
    nome_serie="Concessões - Crédito pessoal não consignado PF",
    data_inicial="01/01/2024",
    data_final="31/12/2024"
)

concessoes_teste

,data,valor,serie,codigo_serie
0,2024-01-01,18200,Concessões - Crédito pessoal não consignado PF,20666
1,2024-02-01,16949,Concessões - Crédito pessoal não consignado PF,20666
2,2024-03-01,16536,Concessões - Crédito pessoal não consignado PF,20666
3,2024-04-01,18552,Concessões - Crédito pessoal não consignado PF,20666
4,2024-05-01,19029,Concessões - Crédito pessoal não consignado PF,20666
5,2024-06-01,17003,Concessões - Crédito pessoal não consignado PF,20666
6,2024-07-01,20201,Concessões - Crédito pessoal não consignado PF,20666
7,2024-08-01,20170,Concessões - Crédito pessoal não consignado PF,20666
8,2024-09-01,19946,Concessões - Crédito pessoal não consignado PF,20666
9,2024-10-01,22033,Concessões - Crédito pessoal não consignado PF,20666


In [6]:
print(concessoes_teste.info())

print("\nPeríodo inicial:")
print(concessoes_teste["data"].min())

print("\nPeríodo final:")
print(concessoes_teste["data"].max())

print("\nValores ausentes:")
print(concessoes_teste.isna().sum())

print("\nDatas duplicadas:")
print(concessoes_teste["data"].duplicated().sum())

print("\nQuantidade de registros:")
print(len(concessoes_teste))

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          12 non-null     datetime64[us]
 1   valor         12 non-null     int64         
 2   serie         12 non-null     str           
 3   codigo_serie  12 non-null     int64         
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 1.1 KB
None

Período inicial:
2024-01-01 00:00:00

Período final:
2024-12-01 00:00:00

Valores ausentes:
data            0
valor           0
serie           0
codigo_serie    0
dtype: int64

Datas duplicadas:
0

Quantidade de registros:
12


In [7]:
def buscar_serie_bcb_em_blocos(
    codigo_serie,
    nome_serie,
    data_inicial,
    data_final,
    anos_por_bloco=9
):
    inicio = pd.to_datetime(data_inicial, format="%d/%m/%Y")
    fim = pd.to_datetime(data_final, format="%d/%m/%Y")

    blocos = []
    inicio_bloco = inicio

    while inicio_bloco <= fim:
        fim_bloco = min(
            inicio_bloco + pd.DateOffset(years=anos_por_bloco)
            - pd.Timedelta(days=1),
            fim
        )

        print(
            f"Buscando de {inicio_bloco:%d/%m/%Y} "
            f"até {fim_bloco:%d/%m/%Y}..."
        )

        df_bloco = buscar_serie_bcb(
            codigo_serie=codigo_serie,
            nome_serie=nome_serie,
            data_inicial=inicio_bloco.strftime("%d/%m/%Y"),
            data_final=fim_bloco.strftime("%d/%m/%Y")
        )

        blocos.append(df_bloco)
        inicio_bloco = fim_bloco + pd.Timedelta(days=1)

    df_completo = pd.concat(blocos, ignore_index=True)

    return (
        df_completo
        .sort_values("data")
        .drop_duplicates(subset="data")
        .reset_index(drop=True)
    )

In [8]:
concessoes = buscar_serie_bcb_em_blocos(
    codigo_serie=20666,
    nome_serie="Concessões - Crédito pessoal não consignado PF",
    data_inicial="01/01/2011",
    data_final="31/12/2025"
)

concessoes

Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...


,data,valor,serie,codigo_serie
0,2011-03-01,6198,Concessões - Crédito pessoal não consignado PF,20666
1,2011-04-01,5871,Concessões - Crédito pessoal não consignado PF,20666
2,2011-05-01,6663,Concessões - Crédito pessoal não consignado PF,20666
3,2011-06-01,6786,Concessões - Crédito pessoal não consignado PF,20666
4,2011-07-01,6327,Concessões - Crédito pessoal não consignado PF,20666
...,...,...,...,...
173,2025-08-01,21591,Concessões - Crédito pessoal não consignado PF,20666
174,2025-09-01,24386,Concessões - Crédito pessoal não consignado PF,20666
175,2025-10-01,25495,Concessões - Crédito pessoal não consignado PF,20666
176,2025-11-01,19821,Concessões - Crédito pessoal não consignado PF,20666


In [9]:
concessoes.info()

print("\nPeríodo inicial:")
print(concessoes["data"].min())

print("\nPeríodo final:")
print(concessoes["data"].max())

print("\nQuantidade de registros:")
print(len(concessoes))

print("\nValores ausentes:")
print(concessoes.isna().sum())

print("\nDatas duplicadas:")
print(concessoes["data"].duplicated().sum())

# Verificar se existem meses ausentes
datas_esperadas = pd.date_range(
    start=concessoes["data"].min(),
    end=concessoes["data"].max(),
    freq="MS"
)

meses_ausentes = datas_esperadas.difference(concessoes["data"])

print("\nMeses ausentes:")
print(meses_ausentes)

print("\nResumo dos valores:")
print(concessoes["valor"].describe())

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   data          178 non-null    datetime64[us]
 1   valor         178 non-null    int64         
 2   serie         178 non-null    str           
 3   codigo_serie  178 non-null    int64         
dtypes: datetime64[us](1), int64(2), str(1)
memory usage: 14.2 KB

Período inicial:
2011-03-01 00:00:00

Período final:
2025-12-01 00:00:00

Quantidade de registros:
178

Valores ausentes:
data            0
valor           0
serie           0
codigo_serie    0
dtype: int64

Datas duplicadas:
0

Meses ausentes:
DatetimeIndex([], dtype='datetime64[us]', freq='MS')

Resumo dos valores:
count      178.000000
mean     10786.432584
std       4955.388039
min       5618.000000
25%       6887.750000
50%       8331.500000
75%      14451.750000
max      25495.000000
Name: valor, dtype: float64


In [10]:
configuracao_series = {
    "inadimplencia_nao_consignado": {
        "codigo": 21114,
        "nome": "Inadimplência - Crédito pessoal não consignado PF"
    },
    "juros_nao_consignado": {
        "codigo": 25464,
        "nome": "Taxa de juros - Crédito pessoal não consignado PF"
    },
    "saldo_nao_consignado": {
        "codigo": 20574,
        "nome": "Saldo - Crédito pessoal não consignado PF"
    },
    "inadimplencia_consignado": {
        "codigo": 21119,
        "nome": "Inadimplência - Crédito pessoal consignado PF"
    },
    "selic": {
        "codigo": 4390,
        "nome": "Selic acumulada no mês"
    },
    "ipca": {
        "codigo": 433,
        "nome": "IPCA mensal"
    }
}

In [11]:
bases = {
    "concessoes": concessoes
}

for chave, configuracao in configuracao_series.items():
    print(f"\nColetando: {configuracao['nome']}")

    bases[chave] = buscar_serie_bcb_em_blocos(
        codigo_serie=configuracao["codigo"],
        nome_serie=configuracao["nome"],
        data_inicial="01/01/2011",
        data_final="31/12/2025"
    )


Coletando: Inadimplência - Crédito pessoal não consignado PF
Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...

Coletando: Taxa de juros - Crédito pessoal não consignado PF
Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...

Coletando: Saldo - Crédito pessoal não consignado PF
Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...

Coletando: Inadimplência - Crédito pessoal consignado PF
Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...

Coletando: Selic acumulada no mês
Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...

Coletando: IPCA mensal
Buscando de 01/01/2011 até 31/12/2019...
Buscando de 01/01/2020 até 31/12/2025...


In [12]:
resumo_coleta = pd.DataFrame([
    {
        "base": nome,
        "registros": len(df),
        "inicio": df["data"].min(),
        "fim": df["data"].max(),
        "valores_ausentes": df["valor"].isna().sum(),
        "datas_duplicadas": df["data"].duplicated().sum()
    }
    for nome, df in bases.items()
])

resumo_coleta

,base,registros,inicio,fim,valores_ausentes,datas_duplicadas
0,concessoes,178,2011-03-01,2025-12-01,0,0
1,inadimplencia_nao_consignado,178,2011-03-01,2025-12-01,0,0
2,juros_nao_consignado,178,2011-03-01,2025-12-01,0,0
3,saldo_nao_consignado,180,2011-01-01,2025-12-01,0,0
4,inadimplencia_consignado,178,2011-03-01,2025-12-01,0,0
5,selic,180,2011-01-01,2025-12-01,0,0
6,ipca,180,2011-01-01,2025-12-01,0,0


In [13]:
nomes_colunas = {
    "concessoes": "concessoes_milhoes",
    "inadimplencia_nao_consignado": "inadimplencia_nao_consignado_pct",
    "juros_nao_consignado": "juros_nao_consignado_pct_mes",
    "saldo_nao_consignado": "saldo_nao_consignado_milhoes",
    "inadimplencia_consignado": "inadimplencia_consignado_pct",
    "selic": "selic_pct_mes",
    "ipca": "ipca_pct_mes"
}

base_analitica = None

for nome_base, nome_coluna in nomes_colunas.items():
    tabela_temporaria = (
        bases[nome_base][["data", "valor"]]
        .rename(columns={"valor": nome_coluna})
    )

    if base_analitica is None:
        base_analitica = tabela_temporaria
    else:
        base_analitica = base_analitica.merge(
            tabela_temporaria,
            on="data",
            how="inner"
        )

base_analitica = (
    base_analitica
    .sort_values("data")
    .reset_index(drop=True)
)

base_analitica.head()

,data,concessoes_milhoes,inadimplencia_nao_consignado_pct,juros_nao_consignado_pct_mes,saldo_nao_consignado_milhoes,inadimplencia_consignado_pct,selic_pct_mes,ipca_pct_mes
0,2011-03-01,6198,6.90,4.72,67845,2.73,0.92,0.79
1,2011-04-01,5871,7.09,4.86,69215,2.73,0.84,0.77
2,2011-05-01,6663,7.44,4.89,70433,2.76,0.99,0.47
3,2011-06-01,6786,7.38,4.90,72007,2.74,0.96,0.15
4,2011-07-01,6327,7.62,5.03,72312,2.76,0.97,0.16


In [14]:
print("Formato da base:", base_analitica.shape)

print("\nPeríodo:")
print(base_analitica["data"].min())
print(base_analitica["data"].max())

print("\nValores ausentes:")
print(base_analitica.isna().sum())

base_analitica.tail()

Formato da base: (178, 8)

Período:
2011-03-01 00:00:00
2025-12-01 00:00:00

Valores ausentes:
data                                0
concessoes_milhoes                  0
inadimplencia_nao_consignado_pct    0
juros_nao_consignado_pct_mes        0
saldo_nao_consignado_milhoes        0
inadimplencia_consignado_pct        0
selic_pct_mes                       0
ipca_pct_mes                        0
dtype: int64


,data,concessoes_milhoes,inadimplencia_nao_consignado_pct,juros_nao_consignado_pct_mes,saldo_nao_consignado_milhoes,inadimplencia_consignado_pct,selic_pct_mes,ipca_pct_mes
173,2025-08-01,21591,9.13,6.12,369249,2.61,1.16,-0.11
174,2025-09-01,24386,8.78,5.99,373360,2.57,1.22,0.48
175,2025-10-01,25495,8.92,6.00,382612,2.61,1.28,0.09
176,2025-11-01,19821,8.91,6.44,390071,2.62,1.05,0.18
177,2025-12-01,21753,9.16,6.65,388413,2.81,1.22,0.33


In [15]:
# Preservar a base original
base_metricas = base_analitica.copy()

# Média móvel de 3 meses das concessões
base_metricas["concessoes_media_movel_3m"] = (
    base_metricas["concessoes_milhoes"]
    .rolling(window=3, min_periods=3)
    .mean()
)

# Variação das concessões contra 12 meses antes
base_metricas["concessoes_var_12m_pct"] = (
    base_metricas["concessoes_media_movel_3m"]
    .pct_change(periods=12, fill_method=None)
    .mul(100)
)

# Variação do saldo da carteira em 12 meses
base_metricas["saldo_var_12m_pct"] = (
    base_metricas["saldo_nao_consignado_milhoes"]
    .pct_change(periods=12, fill_method=None)
    .mul(100)
)

# Variação da inadimplência em 12 meses
base_metricas["inadimplencia_var_12m_pp"] = (
    base_metricas["inadimplencia_nao_consignado_pct"]
    - base_metricas["inadimplencia_nao_consignado_pct"].shift(12)
)

# Diferença entre não consignado e consignado
base_metricas["diferencial_inadimplencia_pp"] = (
    base_metricas["inadimplencia_nao_consignado_pct"]
    - base_metricas["inadimplencia_consignado_pct"]
)

# Prêmio bruto da taxa do crédito sobre a Selic
base_metricas["premio_sobre_selic_pp"] = (
    base_metricas["juros_nao_consignado_pct_mes"]
    - base_metricas["selic_pct_mes"]
)

In [16]:
colunas_exibicao = [
    "data",
    "concessoes_milhoes",
    "concessoes_var_12m_pct",
    "saldo_var_12m_pct",
    "inadimplencia_nao_consignado_pct",
    "inadimplencia_var_12m_pp",
    "diferencial_inadimplencia_pp",
    "premio_sobre_selic_pp"
]

base_metricas[colunas_exibicao].tail(12).round(2)

C:\Users\migue\AppData\Local\Temp\ipykernel_13168\793715666.py:12: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  base_metricas[colunas_exibicao].tail(12).round(2)


,data,concessoes_milhoes,concessoes_var_12m_pct,saldo_var_12m_pct,inadimplencia_nao_consignado_pct,inadimplencia_var_12m_pp,diferencial_inadimplencia_pp,premio_sobre_selic_pp
166,2025-01-01,22880,30.64,21.26,7.09,0.46,4.66,4.93
167,2025-02-01,20026,28.07,21.11,7.37,0.88,5.00,5.23
168,2025-03-01,20057,21.82,20.55,7.32,0.88,4.96,5.22
169,2025-04-01,20788,16.98,21.65,7.67,1.19,5.27,5.14
170,2025-05-01,21364,14.95,20.52,7.91,1.38,5.50,4.99
171,2025-06-01,21169,16.01,21.18,8.25,1.77,5.78,5.21
172,2025-07-01,23002,16.54,21.61,8.66,2.07,6.09,4.87
173,2025-08-01,21591,14.62,20.38,9.13,2.63,6.52,4.96
174,2025-09-01,24386,14.36,19.04,8.78,2.29,6.21,4.77
175,2025-10-01,25495,15.00,19.04,8.92,2.51,6.31,4.72


In [17]:
resultado_metricas = (
    base_metricas[colunas_exibicao]
    .tail(12)
    .copy()
)

colunas_numericas = resultado_metricas.select_dtypes(
    include="number"
).columns

resultado_metricas[colunas_numericas] = (
    resultado_metricas[colunas_numericas].round(2)
)

resultado_metricas

,data,concessoes_milhoes,concessoes_var_12m_pct,saldo_var_12m_pct,inadimplencia_nao_consignado_pct,inadimplencia_var_12m_pp,diferencial_inadimplencia_pp,premio_sobre_selic_pp
166,2025-01-01,22880,30.64,21.26,7.09,0.46,4.66,4.93
167,2025-02-01,20026,28.07,21.11,7.37,0.88,5.00,5.23
168,2025-03-01,20057,21.82,20.55,7.32,0.88,4.96,5.22
169,2025-04-01,20788,16.98,21.65,7.67,1.19,5.27,5.14
170,2025-05-01,21364,14.95,20.52,7.91,1.38,5.50,4.99
171,2025-06-01,21169,16.01,21.18,8.25,1.77,5.78,5.21
172,2025-07-01,23002,16.54,21.61,8.66,2.07,6.09,4.87
173,2025-08-01,21591,14.62,20.38,9.13,2.63,6.52,4.96
174,2025-09-01,24386,14.36,19.04,8.78,2.29,6.21,4.77
175,2025-10-01,25495,15.00,19.04,8.92,2.51,6.31,4.72


In [18]:
colunas_calculadas = [
    "concessoes_media_movel_3m",
    "concessoes_var_12m_pct",
    "saldo_var_12m_pct",
    "inadimplencia_var_12m_pp",
    "diferencial_inadimplencia_pp",
    "premio_sobre_selic_pp"
]

print("Valores ausentes por indicador:")
print(base_metricas[colunas_calculadas].isna().sum())

print("\nValores infinitos:")
print(
    base_metricas[colunas_calculadas]
    .isin([float("inf"), float("-inf")])
    .sum()
)

Valores ausentes por indicador:
concessoes_media_movel_3m        2
concessoes_var_12m_pct          14
saldo_var_12m_pct               12
inadimplencia_var_12m_pp        12
diferencial_inadimplencia_pp     0
premio_sobre_selic_pp            0
dtype: int64

Valores infinitos:
concessoes_media_movel_3m       0
concessoes_var_12m_pct          0
saldo_var_12m_pct               0
inadimplencia_var_12m_pp        0
diferencial_inadimplencia_pp    0
premio_sobre_selic_pp           0
dtype: int64


In [19]:
metricas_cenario = (
    base_metricas[
        [
            "concessoes_var_12m_pct",
            "inadimplencia_var_12m_pp"
        ]
    ]
    .dropna()
    .copy()
)

resumo_distribuicao = metricas_cenario.describe(
    percentiles=[0.10, 0.20, 0.25, 0.50, 0.75, 0.80, 0.90]
).T

resumo_distribuicao.round(2)

,count,mean,std,min,10%,20%,25%,50%,75%,80%,90%,max
concessoes_var_12m_pct,164.0,9.98,15.06,-11.35,-7.68,-4.33,-2.22,6.58,19.97,23.04,32.37,48.61
inadimplencia_var_12m_pp,164.0,0.01,1.51,-3.29,-1.78,-1.37,-1.06,0.00,1.03,1.36,2.28,3.09


In [20]:
movimentos_absolutos = metricas_cenario.abs()

limites_candidatos = pd.DataFrame({
    "percentil_10": movimentos_absolutos.quantile(0.10),
    "percentil_20": movimentos_absolutos.quantile(0.20),
    "percentil_25": movimentos_absolutos.quantile(0.25)
})

limites_candidatos.round(2)

,percentil_10,percentil_20,percentil_25
concessoes_var_12m_pct,1.84,3.30,4.40
inadimplencia_var_12m_pp,0.28,0.46,0.53


In [21]:
limite_concessoes = movimentos_absolutos[
    "concessoes_var_12m_pct"
].quantile(0.20)

limite_inadimplencia = movimentos_absolutos[
    "inadimplencia_var_12m_pp"
].quantile(0.20)

print(
    "Limite neutro das concessões:",
    round(limite_concessoes, 2),
    "%"
)

print(
    "Limite neutro da inadimplência:",
    round(limite_inadimplencia, 2),
    "p.p."
)

Limite neutro das concessões: 3.3 %
Limite neutro da inadimplência: 0.46 p.p.


In [22]:
def classificar_movimento(valor, limite):
    if pd.isna(valor):
        return "Sem histórico"

    if valor > limite:
        return "Alta"

    if valor < -limite:
        return "Queda"

    return "Estável"


base_metricas["movimento_concessoes"] = (
    base_metricas["concessoes_var_12m_pct"]
    .apply(
        lambda valor: classificar_movimento(
            valor,
            limite_concessoes
        )
    )
)

base_metricas["movimento_inadimplencia"] = (
    base_metricas["inadimplencia_var_12m_pp"]
    .apply(
        lambda valor: classificar_movimento(
            valor,
            limite_inadimplencia
        )
    )
)

In [23]:
def classificar_cenario(linha):
    concessoes = linha["movimento_concessoes"]
    inadimplencia = linha["movimento_inadimplencia"]

    if "Sem histórico" in [concessoes, inadimplencia]:
        return "Sem histórico suficiente"

    if (
        concessoes == "Alta"
        and inadimplencia in ["Queda", "Estável"]
    ):
        return "Expansão sem deterioração observada"

    if (
        concessoes == "Alta"
        and inadimplencia == "Alta"
    ):
        return "Crescimento com alerta"

    if (
        concessoes == "Queda"
        and inadimplencia == "Alta"
    ):
        return "Aperto com deterioração"

    if (
        concessoes in ["Queda", "Estável"]
        and inadimplencia == "Queda"
    ):
        return "Recuperação do risco"

    return "Transição/estabilidade"


base_metricas["cenario"] = base_metricas.apply(
    classificar_cenario,
    axis=1
)

In [24]:
print("Quantidade de meses por cenário:")

print(
    base_metricas["cenario"]
    .value_counts()
)

colunas_cenario = [
    "data",
    "concessoes_var_12m_pct",
    "inadimplencia_var_12m_pp",
    "movimento_concessoes",
    "movimento_inadimplencia",
    "cenario"
]

base_metricas[colunas_cenario].tail(12)

Quantidade de meses por cenário:
cenario
Expansão sem deterioração observada    66
Crescimento com alerta                 28
Recuperação do risco                   25
Aperto com deterioração                23
Transição/estabilidade                 22
Sem histórico suficiente               14
Name: count, dtype: int64


,data,concessoes_var_12m_pct,inadimplencia_var_12m_pp,movimento_concessoes,movimento_inadimplencia,cenario
166,2025-01-01,30.640298,0.46,Alta,Estável,Expansão sem deterioração observada
167,2025-02-01,28.070671,0.88,Alta,Alta,Crescimento com alerta
168,2025-03-01,21.820644,0.88,Alta,Alta,Crescimento com alerta
169,2025-04-01,16.976382,1.19,Alta,Alta,Crescimento com alerta
170,2025-05-01,14.952787,1.38,Alta,Alta,Crescimento com alerta
171,2025-06-01,16.006522,1.77,Alta,Alta,Crescimento com alerta
172,2025-07-01,16.541888,2.07,Alta,Alta,Crescimento com alerta
173,2025-08-01,14.619863,2.63,Alta,Alta,Crescimento com alerta
174,2025-09-01,14.360794,2.29,Alta,Alta,Crescimento com alerta
175,2025-10-01,15.001046,2.51,Alta,Alta,Crescimento com alerta


In [25]:
import plotly.express as px

dados_matriz = (
    base_metricas
    .dropna(
        subset=[
            "concessoes_var_12m_pct",
            "inadimplencia_var_12m_pp"
        ]
    )
    .copy()
)

dados_matriz["periodo"] = (
    dados_matriz["data"].dt.strftime("%m/%Y")
)

cores_cenarios = {
    "Expansão sem deterioração observada": "#2E8B57",
    "Crescimento com alerta": "#F4A261",
    "Aperto com deterioração": "#D62828",
    "Recuperação do risco": "#277DA1",
    "Transição/estabilidade": "#9E9E9E"
}

fig = px.scatter(
    dados_matriz,
    x="concessoes_var_12m_pct",
    y="inadimplencia_var_12m_pp",
    color="cenario",
    color_discrete_map=cores_cenarios,
    hover_name="periodo",
    hover_data={
        "concessoes_var_12m_pct": ":.2f",
        "inadimplencia_var_12m_pp": ":.2f"
    },
    labels={
        "concessoes_var_12m_pct":
            "Variação das concessões em 12 meses (%)",
        "inadimplencia_var_12m_pp":
            "Variação da inadimplência em 12 meses (p.p.)",
        "cenario": "Cenário"
    },
    title="Matriz de cenários do crédito pessoal não consignado"
)

# Faixas consideradas neutras
fig.add_vrect(
    x0=-limite_concessoes,
    x1=limite_concessoes,
    fillcolor="gray",
    opacity=0.08,
    line_width=0
)

fig.add_hrect(
    y0=-limite_inadimplencia,
    y1=limite_inadimplencia,
    fillcolor="gray",
    opacity=0.08,
    line_width=0
)

# Linhas centrais
fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="gray"
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

fig.update_layout(
    template="plotly_white",
    legend_title_text="Cenário",
    height=650
)

fig.show()

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_temporal = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=[
        "Concessões mensais",
        "Inadimplência: não consignado versus consignado"
    ]
)

# Concessões mensais
fig_temporal.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["concessoes_milhoes"],
        name="Concessões mensais",
        line={
            "color": "#AAB7C4",
            "width": 1
        },
        opacity=0.70
    ),
    row=1,
    col=1
)

# Média móvel das concessões
fig_temporal.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["concessoes_media_movel_3m"],
        name="Média móvel de 3 meses",
        line={
            "color": "#1F77B4",
            "width": 3
        }
    ),
    row=1,
    col=1
)

# Inadimplência não consignado
fig_temporal.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["inadimplencia_nao_consignado_pct"],
        name="Não consignado",
        line={
            "color": "#D62828",
            "width": 3
        }
    ),
    row=2,
    col=1
)

# Inadimplência consignado
fig_temporal.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["inadimplencia_consignado_pct"],
        name="Consignado",
        line={
            "color": "#277DA1",
            "width": 2
        }
    ),
    row=2,
    col=1
)

# Destacar o período da pandemia
fig_temporal.add_vrect(
    x0="2020-03-01",
    x1="2021-12-31",
    fillcolor="#F4A261",
    opacity=0.12,
    line_width=0,
    annotation_text="Período da pandemia",
    annotation_position="top left",
    row="all",
    col=1
)

fig_temporal.update_yaxes(
    title_text="R$ milhões",
    row=1,
    col=1
)

fig_temporal.update_yaxes(
    title_text="Inadimplência (%)",
    row=2,
    col=1
)

fig_temporal.update_xaxes(
    title_text="Período",
    row=2,
    col=1
)

fig_temporal.update_layout(
    title="Evolução das concessões e da inadimplência",
    template="plotly_white",
    height=800,
    hovermode="x unified",
    legend_title_text="Indicador"
)

fig_temporal.show()

In [27]:
fig_juros = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=[
        "Taxa do crédito pessoal não consignado e Selic",
        "Prêmio bruto da taxa do crédito sobre a Selic"
    ]
)

# Taxa do crédito
fig_juros.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["juros_nao_consignado_pct_mes"],
        name="Taxa do crédito não consignado",
        line={
            "color": "#D62828",
            "width": 3
        }
    ),
    row=1,
    col=1
)

# Selic mensal
fig_juros.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["selic_pct_mes"],
        name="Selic acumulada no mês",
        line={
            "color": "#277DA1",
            "width": 2
        }
    ),
    row=1,
    col=1
)

# Prêmio bruto sobre a Selic
fig_juros.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["premio_sobre_selic_pp"],
        name="Prêmio sobre a Selic",
        fill="tozeroy",
        line={
            "color": "#F4A261",
            "width": 2
        }
    ),
    row=2,
    col=1
)

# Período da pandemia
fig_juros.add_vrect(
    x0="2020-03-01",
    x1="2021-12-31",
    fillcolor="#F4A261",
    opacity=0.12,
    line_width=0,
    annotation_text="Período da pandemia",
    annotation_position="top left",
    row="all",
    col=1
)

fig_juros.update_yaxes(
    title_text="% ao mês",
    row=1,
    col=1
)

fig_juros.update_yaxes(
    title_text="Pontos percentuais",
    row=2,
    col=1
)

fig_juros.update_xaxes(
    title_text="Período",
    row=2,
    col=1
)

fig_juros.update_layout(
    title="Evolução do preço do crédito e da Selic",
    template="plotly_white",
    height=800,
    hovermode="x unified",
    legend_title_text="Indicador"
)

fig_juros.show()

In [28]:
from pathlib import Path

pasta_projeto = Path.cwd()
if pasta_projeto.name == "notebooks":
    pasta_projeto = pasta_projeto.parent

pasta_dados = pasta_projeto / "data" / "processed"
pasta_dados.mkdir(parents=True, exist_ok=True)

# Base original consolidada
base_analitica.to_csv(
    pasta_dados / "base_analitica_bcb.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

# Base com métricas e cenários
base_metricas.to_csv(
    pasta_dados / "base_metricas_bcb.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

print("Tabelas salvas em:")
print(pasta_dados)

Tabelas salvas em:
c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\analise-credito-pessoal\data\processed


In [29]:
pasta_graficos = pasta_projeto / "outputs" / "figures"
pasta_graficos.mkdir(parents=True, exist_ok=True)

fig.write_html(
    pasta_graficos / "matriz_cenarios.html"
)

fig_temporal.write_html(
    pasta_graficos / "concessoes_inadimplencia.html"
)

fig_juros.write_html(
    pasta_graficos / "juros_selic.html"
)

print("Gráficos salvos em:")
print(pasta_graficos)

Gráficos salvos em:
c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\analise-credito-pessoal\outputs\figures


## Ajuste dos valores monetários pelo IPCA

Nesta etapa, as concessões e o saldo da carteira serão convertidos para reais de dezembro de 2025, permitindo separar crescimento nominal de crescimento real.

In [30]:
# Organizar a base por data
base_metricas = (
    base_metricas
    .sort_values("data")
    .reset_index(drop=True)
    .copy()
)

# Construir um índice acumulado de preços
base_metricas["indice_precos_ipca"] = (
    1 + base_metricas["ipca_pct_mes"] / 100
).cumprod()

# Calcular o fator para converter tudo em reais de dezembro de 2025
base_metricas["fator_correcao_dez_2025"] = (
    base_metricas["indice_precos_ipca"].iloc[-1]
    / base_metricas["indice_precos_ipca"]
)

# Corrigir concessões e saldo pela inflação
base_metricas["concessoes_reais_dez_2025_milhoes"] = (
    base_metricas["concessoes_milhoes"]
    * base_metricas["fator_correcao_dez_2025"]
)

base_metricas["saldo_real_dez_2025_milhoes"] = (
    base_metricas["saldo_nao_consignado_milhoes"]
    * base_metricas["fator_correcao_dez_2025"]
)

# Conferir o primeiro e o último mês
colunas_validacao = [
    "data",
    "ipca_pct_mes",
    "fator_correcao_dez_2025",
    "concessoes_milhoes",
    "concessoes_reais_dez_2025_milhoes",
    "saldo_nao_consignado_milhoes",
    "saldo_real_dez_2025_milhoes"
]

display(
    base_metricas[colunas_validacao]
    .iloc[[0, -1]]
    .round(2)
)

C:\Users\migue\AppData\Local\Temp\ipykernel_13168\2837857591.py:45: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,data,ipca_pct_mes,fator_correcao_dez_2025,concessoes_milhoes,concessoes_reais_dez_2025_milhoes,saldo_nao_consignado_milhoes,saldo_real_dez_2025_milhoes
0,2011-03-01,0.79,2.26,6198,14015.76,67845,153420.38
177,2025-12-01,0.33,1.00,21753,21753.00,388413,388413.00


### Comparação entre valores nominais e reais

Os valores reais foram corrigidos pelo IPCA e expressos em preços de dezembro de 2025.

In [31]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_nominal_real = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "Concessões mensais",
        "Saldo da carteira"
    )
)

# Concessões
fig_nominal_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["concessoes_milhoes"],
        name="Concessões nominais",
        line=dict(color="#9BBAD8", width=1.5)
    ),
    row=1,
    col=1
)

fig_nominal_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["concessoes_reais_dez_2025_milhoes"],
        name="Concessões reais",
        line=dict(color="#1565A9", width=2.5)
    ),
    row=1,
    col=1
)

# Saldo da carteira
fig_nominal_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["saldo_nao_consignado_milhoes"],
        name="Saldo nominal",
        line=dict(color="#F4A261", width=1.5)
    ),
    row=2,
    col=1
)

fig_nominal_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["saldo_real_dez_2025_milhoes"],
        name="Saldo real",
        line=dict(color="#D95F02", width=2.5)
    ),
    row=2,
    col=1
)

# Marcação da pandemia
fig_nominal_real.add_vrect(
    x0="2020-03-01",
    x1="2021-12-01",
    fillcolor="#F4A261",
    opacity=0.10,
    line_width=0,
    annotation_text="Período da pandemia",
    annotation_position="top left",
    row="all",
    col="all"
)

fig_nominal_real.update_yaxes(
    title_text="R$ milhões",
    row=1,
    col=1
)

fig_nominal_real.update_yaxes(
    title_text="R$ milhões",
    row=2,
    col=1
)

fig_nominal_real.update_xaxes(
    title_text="Período",
    row=2,
    col=1
)

fig_nominal_real.update_layout(
    title="Valores nominais versus valores reais",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend_title="Indicador"
)

fig_nominal_real.show()

In [32]:
# Média móvel das concessões corrigidas pela inflação
base_metricas["concessoes_reais_media_movel_3m"] = (
    base_metricas["concessoes_reais_dez_2025_milhoes"]
    .rolling(window=3)
    .mean()
)

# Variação real das concessões em 12 meses
base_metricas["concessoes_reais_var_12m_pct"] = (
    base_metricas["concessoes_reais_media_movel_3m"]
    .pct_change(periods=12, fill_method=None)
    * 100
)

# Variação real do saldo em 12 meses
base_metricas["saldo_real_var_12m_pct"] = (
    base_metricas["saldo_real_dez_2025_milhoes"]
    .pct_change(periods=12, fill_method=None)
    * 100
)

# Comparar crescimento nominal e real nos últimos 12 meses
colunas_comparacao = [
    "data",
    "concessoes_var_12m_pct",
    "concessoes_reais_var_12m_pct",
    "saldo_var_12m_pct",
    "saldo_real_var_12m_pct"
]

comparacao_crescimento = (
    base_metricas[colunas_comparacao]
    .tail(12)
    .copy()
)

colunas_numericas = colunas_comparacao[1:]

comparacao_crescimento[colunas_numericas] = (
    comparacao_crescimento[colunas_numericas].round(2)
)

display(comparacao_crescimento)

,data,concessoes_var_12m_pct,concessoes_reais_var_12m_pct,saldo_var_12m_pct,saldo_real_var_12m_pct
166,2025-01-01,30.64,24.72,21.26,15.97
167,2025-02-01,28.07,22.24,21.11,15.28
168,2025-03-01,21.82,16.02,20.55,14.29
169,2025-04-01,16.98,11.04,21.65,15.28
170,2025-05-01,14.95,9.03,20.52,14.43
171,2025-06-01,16.01,10.05,21.18,15.02
172,2025-07-01,16.54,10.68,21.61,15.57
173,2025-08-01,14.62,8.93,20.38,14.51
174,2025-09-01,14.36,8.72,19.04,13.19
175,2025-10-01,15.00,9.53,19.04,13.72


In [33]:
fig_crescimento_real = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "Crescimento das concessões em 12 meses",
        "Crescimento do saldo da carteira em 12 meses"
    )
)

# Concessões
fig_crescimento_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["concessoes_var_12m_pct"],
        name="Concessões: nominal",
        line=dict(color="#9BBAD8", width=1.5)
    ),
    row=1,
    col=1
)

fig_crescimento_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["concessoes_reais_var_12m_pct"],
        name="Concessões: real",
        line=dict(color="#1565A9", width=2.5)
    ),
    row=1,
    col=1
)

# Saldo
fig_crescimento_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["saldo_var_12m_pct"],
        name="Saldo: nominal",
        line=dict(color="#F4A261", width=1.5)
    ),
    row=2,
    col=1
)

fig_crescimento_real.add_trace(
    go.Scatter(
        x=base_metricas["data"],
        y=base_metricas["saldo_real_var_12m_pct"],
        name="Saldo: real",
        line=dict(color="#D95F02", width=2.5)
    ),
    row=2,
    col=1
)

# Linha de crescimento zero
fig_crescimento_real.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    row="all",
    col="all"
)

# Período da pandemia
fig_crescimento_real.add_vrect(
    x0="2020-03-01",
    x1="2021-12-01",
    fillcolor="#F4A261",
    opacity=0.10,
    line_width=0,
    annotation_text="Período da pandemia",
    annotation_position="top left",
    row="all",
    col="all"
)

fig_crescimento_real.update_yaxes(
    title_text="Variação em 12 meses (%)"
)

fig_crescimento_real.update_xaxes(
    title_text="Período",
    row=2,
    col=1
)

fig_crescimento_real.update_layout(
    title="Crescimento nominal versus crescimento real",
    template="plotly_white",
    height=750,
    hovermode="x unified",
    legend_title="Indicador"
)

fig_crescimento_real.show()

### Principais insights da análise real

A correção pelo IPCA mostra que parte relevante do crescimento nominal do crédito correspondeu à inflação. Em diversos períodos, especialmente entre 2012 e 2017, as concessões apresentaram contração em termos reais, embora os valores nominais sugerissem maior estabilidade.

Durante a pandemia, concessões e saldo apresentaram movimentos atípicos. Esse comportamento deve ser interpretado com cautela e não permite concluir causalidade sem informações adicionais.

Em 2025, o crédito continuou crescendo em termos reais. Entretanto, o crescimento real das concessões desacelerou de 24,72% em janeiro para 3,59% em dezembro, enquanto o saldo real encerrou o período com alta de 14,07%.

A combinação entre desaceleração das novas concessões, crescimento persistente do saldo e aumento da inadimplência reforça a classificação de crescimento com alerta.

In [34]:
import numpy as np

colunas_ipca = [
    "indice_precos_ipca",
    "fator_correcao_dez_2025",
    "concessoes_reais_dez_2025_milhoes",
    "saldo_real_dez_2025_milhoes",
    "concessoes_reais_media_movel_3m",
    "concessoes_reais_var_12m_pct",
    "saldo_real_var_12m_pct"
]

print("Valores ausentes:")
display(base_metricas[colunas_ipca].isna().sum())

print("Valores infinitos:")
display(
    np.isinf(
        base_metricas[colunas_ipca].select_dtypes(include="number")
    ).sum()
)

# Validações do mês-base
assert np.isclose(
    base_metricas["fator_correcao_dez_2025"].iloc[-1],
    1
)

assert np.isclose(
    base_metricas["concessoes_milhoes"].iloc[-1],
    base_metricas["concessoes_reais_dez_2025_milhoes"].iloc[-1]
)

assert np.isclose(
    base_metricas["saldo_nao_consignado_milhoes"].iloc[-1],
    base_metricas["saldo_real_dez_2025_milhoes"].iloc[-1]
)

print("Validação do ajuste pelo IPCA concluída com sucesso!")

Valores ausentes:


indice_precos_ipca                    0
fator_correcao_dez_2025               0
concessoes_reais_dez_2025_milhoes     0
saldo_real_dez_2025_milhoes           0
concessoes_reais_media_movel_3m       2
concessoes_reais_var_12m_pct         14
saldo_real_var_12m_pct               12
dtype: int64

Valores infinitos:


indice_precos_ipca                   0
fator_correcao_dez_2025              0
concessoes_reais_dez_2025_milhoes    0
saldo_real_dez_2025_milhoes          0
concessoes_reais_media_movel_3m      0
concessoes_reais_var_12m_pct         0
saldo_real_var_12m_pct               0
dtype: int64

Validação do ajuste pelo IPCA concluída com sucesso!


In [35]:
# Garantir que as pastas existem
pasta_dados_processados = pasta_projeto / "data" / "processed"
pasta_graficos = pasta_projeto / "outputs" / "figures"

pasta_dados_processados.mkdir(parents=True, exist_ok=True)
pasta_graficos.mkdir(parents=True, exist_ok=True)

# Salvar a base com as métricas nominais e reais
arquivo_base_metricas = (
    pasta_dados_processados / "base_metricas_bcb.csv"
)

base_metricas.to_csv(
    arquivo_base_metricas,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

# Salvar os novos gráficos interativos
arquivo_valores_reais = (
    pasta_graficos / "valores_nominais_reais.html"
)

arquivo_crescimento_real = (
    pasta_graficos / "crescimento_nominal_real.html"
)

fig_nominal_real.write_html(
    arquivo_valores_reais,
    include_plotlyjs="cdn"
)

fig_crescimento_real.write_html(
    arquivo_crescimento_real,
    include_plotlyjs="cdn"
)

print("Base atualizada:")
print(arquivo_base_metricas)

print("\nGráficos salvos:")
print(arquivo_valores_reais)
print(arquivo_crescimento_real)

print("\nFormato da base atualizada:")
print(base_metricas.shape)

Base atualizada:
c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\analise-credito-pessoal\data\processed\base_metricas_bcb.csv

Gráficos salvos:
c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\analise-credito-pessoal\outputs\figures\valores_nominais_reais.html
c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\analise-credito-pessoal\outputs\figures\crescimento_nominal_real.html

Formato da base atualizada:
(178, 24)


## Linha do tempo dos cenários

Esta visualização apresenta a classificação mensal do crédito pessoal não consignado, combinando o movimento das concessões com a evolução da inadimplência.

In [36]:
import plotly.express as px

# Retirar os meses sem histórico suficiente
base_cenarios_tempo = base_metricas[
    base_metricas["cenario"] != "Sem histórico suficiente"
].copy()

# Definir cores dos cenários
cores_cenarios = {
    "Expansão sem deterioração observada": "#2E8B57",
    "Crescimento com alerta": "#F4A261",
    "Aperto com deterioração": "#D62828",
    "Recuperação do risco": "#2C7FB8",
    "Transição/estabilidade": "#A0A0A0"
}

ordem_cenarios = [
    "Expansão sem deterioração observada",
    "Crescimento com alerta",
    "Transição/estabilidade",
    "Recuperação do risco",
    "Aperto com deterioração"
]

fig_linha_tempo = px.scatter(
    base_cenarios_tempo,
    x="data",
    y="cenario",
    color="cenario",
    color_discrete_map=cores_cenarios,
    category_orders={"cenario": ordem_cenarios},
    hover_name="cenario",
    hover_data={
        "data": "|%b/%Y",
        "concessoes_var_12m_pct": ":.2f",
        "inadimplencia_var_12m_pp": ":.2f",
        "cenario": False
    },
    labels={
        "data": "Período",
        "cenario": "Cenário",
        "concessoes_var_12m_pct": "Concessões em 12 meses (%)",
        "inadimplencia_var_12m_pp": "Inadimplência em 12 meses (p.p.)"
    },
    title="Linha do tempo dos cenários de crédito"
)

fig_linha_tempo.update_traces(
    marker=dict(size=9, opacity=0.90)
)

# Destacar a pandemia
fig_linha_tempo.add_vrect(
    x0="2020-03-01",
    x1="2021-12-01",
    fillcolor="#F4A261",
    opacity=0.10,
    line_width=0,
    annotation_text="Período da pandemia",
    annotation_position="top left"
)

fig_linha_tempo.update_layout(
    template="plotly_white",
    height=550,
    hovermode="closest",
    legend_title="Cenário"
)

fig_linha_tempo.show()

In [37]:
arquivo_linha_tempo = (
    pasta_graficos / "linha_tempo_cenarios.html"
)

fig_linha_tempo.write_html(
    arquivo_linha_tempo,
    include_plotlyjs="cdn"
)

print("Linha do tempo salva em:")
print(arquivo_linha_tempo)

Linha do tempo salva em:
c:\Users\migue\OneDrive\Área de Trabalho\Cases Python\Projeto Crédito Pessoal\analise-credito-pessoal\analise-credito-pessoal\outputs\figures\linha_tempo_cenarios.html


## Teste de sensibilidade dos cenários

A classificação original utiliza o crescimento nominal das concessões. Nesta etapa, a metodologia será repetida usando o crescimento real corrigido pelo IPCA, permitindo verificar quais meses mudam de cenário.

In [38]:
# Calcular os limites pelo percentil 20
limite_concessoes_real = (
    base_metricas["concessoes_reais_var_12m_pct"]
    .dropna()
    .abs()
    .quantile(0.20)
)

limite_inadimplencia_teste = (
    base_metricas["inadimplencia_var_12m_pp"]
    .dropna()
    .abs()
    .quantile(0.20)
)

print(
    "Limite das concessões reais:",
    round(limite_concessoes_real, 2),
    "%"
)

print(
    "Limite da inadimplência:",
    round(limite_inadimplencia_teste, 2),
    "p.p."
)

Limite das concessões reais: 4.44 %
Limite da inadimplência: 0.46 p.p.


In [39]:
def classificar_movimento(valor, limite):
    if pd.isna(valor):
        return "Sem histórico suficiente"
    elif valor > limite:
        return "Alta"
    elif valor < -limite:
        return "Queda"
    else:
        return "Estável"


base_metricas["movimento_concessoes_real"] = (
    base_metricas["concessoes_reais_var_12m_pct"]
    .apply(
        lambda valor: classificar_movimento(
            valor,
            limite_concessoes_real
        )
    )
)


def classificar_cenario_real(linha):
    concessoes = linha["movimento_concessoes_real"]
    inadimplencia = linha["movimento_inadimplencia"]

    if (
        pd.isna(linha["concessoes_reais_var_12m_pct"])
        or pd.isna(linha["inadimplencia_var_12m_pp"])
    ):
        return "Sem histórico suficiente"

    if (
        concessoes == "Alta"
        and inadimplencia in ["Estável", "Queda"]
    ):
        return "Expansão sem deterioração observada"

    elif (
        concessoes == "Alta"
        and inadimplencia == "Alta"
    ):
        return "Crescimento com alerta"

    elif (
        concessoes == "Queda"
        and inadimplencia == "Alta"
    ):
        return "Aperto com deterioração"

    elif (
        concessoes in ["Queda", "Estável"]
        and inadimplencia == "Queda"
    ):
        return "Recuperação do risco"

    else:
        return "Transição/estabilidade"


base_metricas["cenario_real"] = base_metricas.apply(
    classificar_cenario_real,
    axis=1
)

In [40]:
comparaveis = base_metricas[
    (base_metricas["cenario"] != "Sem histórico suficiente")
    & (
        base_metricas["cenario_real"]
        != "Sem histórico suficiente"
    )
].copy()

comparaveis["cenario_alterado"] = (
    comparaveis["cenario"]
    != comparaveis["cenario_real"]
)

quantidade_alterada = comparaveis["cenario_alterado"].sum()
percentual_alterado = comparaveis["cenario_alterado"].mean() * 100

print("Meses comparáveis:", len(comparaveis))
print("Meses que mudaram:", quantidade_alterada)
print(
    "Percentual que mudou:",
    round(percentual_alterado, 2),
    "%"
)

tabela_mudancas = comparaveis[
    comparaveis["cenario_alterado"]
][
    [
        "data",
        "concessoes_var_12m_pct",
        "concessoes_reais_var_12m_pct",
        "inadimplencia_var_12m_pp",
        "cenario",
        "cenario_real"
    ]
].copy()

colunas_numericas = [
    "concessoes_var_12m_pct",
    "concessoes_reais_var_12m_pct",
    "inadimplencia_var_12m_pp"
]

tabela_mudancas[colunas_numericas] = (
    tabela_mudancas[colunas_numericas].round(2)
)

display(tabela_mudancas.tail(20))

Meses comparáveis: 164
Meses que mudaram: 28
Percentual que mudou: 17.07 %


,data,concessoes_var_12m_pct,concessoes_reais_var_12m_pct,inadimplencia_var_12m_pp,cenario,cenario_real
49,2015-04-01,8.82,0.70,0.42,Expansão sem deterioração observada,Transição/estabilidade
50,2015-05-01,10.40,2.03,0.43,Expansão sem deterioração observada,Transição/estabilidade
51,2015-06-01,6.76,-1.62,-0.02,Expansão sem deterioração observada,Transição/estabilidade
52,2015-07-01,5.45,-3.24,0.05,Expansão sem deterioração observada,Transição/estabilidade
53,2015-08-01,5.44,-3.54,0.45,Expansão sem deterioração observada,Transição/estabilidade
54,2015-09-01,-2.20,-10.70,0.74,Transição/estabilidade,Aperto com deterioração
59,2016-02-01,-3.21,-12.47,2.30,Transição/estabilidade,Aperto com deterioração
76,2017-07-01,5.92,2.72,-1.62,Expansão sem deterioração observada,Recuperação do risco
77,2017-08-01,5.16,2.36,-1.36,Expansão sem deterioração observada,Recuperação do risco
113,2020-08-01,4.12,1.81,-0.61,Expansão sem deterioração observada,Recuperação do risco


In [41]:
# Resumo da sensibilidade
print(
    "Limite nominal das concessões:",
    round(limite_concessoes, 2),
    "%"
)

print(
    "Limite real das concessões:",
    round(limite_concessoes_real, 2),
    "%"
)

print(
    "Limite da inadimplência:",
    round(limite_inadimplencia_teste, 2),
    "p.p."
)

print("\nMeses comparáveis:", len(comparaveis))
print("Meses alterados:", quantidade_alterada)
print(
    "Percentual alterado:",
    round(percentual_alterado, 2),
    "%"
)

# Contar os cenários
resumo_nominal = (
    comparaveis["cenario"]
    .value_counts()
    .rename("meses_nominal")
)

resumo_real = (
    comparaveis["cenario_real"]
    .value_counts()
    .rename("meses_real")
)

resumo_cenarios = (
    pd.concat(
        [resumo_nominal, resumo_real],
        axis=1
    )
    .fillna(0)
    .astype(int)
)

resumo_cenarios["diferenca_meses"] = (
    resumo_cenarios["meses_real"]
    - resumo_cenarios["meses_nominal"]
)

resumo_cenarios["participacao_real_pct"] = (
    resumo_cenarios["meses_real"]
    / len(comparaveis)
    * 100
).round(2)

resumo_cenarios = (
    resumo_cenarios
    .reindex(ordem_cenarios)
    .fillna(0)
)

display(resumo_cenarios)

Limite nominal das concessões: 3.3 %
Limite real das concessões: 4.44 %
Limite da inadimplência: 0.46 p.p.

Meses comparáveis: 164
Meses alterados: 28
Percentual alterado: 17.07 %


,meses_nominal,meses_real,diferenca_meses,participacao_real_pct
Expansão sem deterioração observada,66,50,-16,30.49
Crescimento com alerta,28,22,-6,13.41
Transição/estabilidade,22,28,6,17.07
Recuperação do risco,25,35,10,21.34
Aperto com deterioração,23,29,6,17.68


In [42]:
revisao_recente = base_metricas[
    [
        "data",
        "concessoes_var_12m_pct",
        "concessoes_reais_var_12m_pct",
        "inadimplencia_var_12m_pp",
        "movimento_concessoes",
        "movimento_concessoes_real",
        "movimento_inadimplencia",
        "cenario",
        "cenario_real"
    ]
].tail(12).copy()

colunas_numericas = [
    "concessoes_var_12m_pct",
    "concessoes_reais_var_12m_pct",
    "inadimplencia_var_12m_pp"
]

revisao_recente[colunas_numericas] = (
    revisao_recente[colunas_numericas].round(2)
)

display(revisao_recente)

,data,concessoes_var_12m_pct,concessoes_reais_var_12m_pct,inadimplencia_var_12m_pp,movimento_concessoes,movimento_concessoes_real,movimento_inadimplencia,cenario,cenario_real
166,2025-01-01,30.64,24.72,0.46,Alta,Alta,Estável,Expansão sem deterioração observada,Expansão sem deterioração observada
167,2025-02-01,28.07,22.24,0.88,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
168,2025-03-01,21.82,16.02,0.88,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
169,2025-04-01,16.98,11.04,1.19,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
170,2025-05-01,14.95,9.03,1.38,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
171,2025-06-01,16.01,10.05,1.77,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
172,2025-07-01,16.54,10.68,2.07,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
173,2025-08-01,14.62,8.93,2.63,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
174,2025-09-01,14.36,8.72,2.29,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta
175,2025-10-01,15.00,9.53,2.51,Alta,Alta,Alta,Crescimento com alerta,Crescimento com alerta


### Refinamento da classificação

O teste com valores reais revelou que períodos com concessões estáveis e inadimplência crescente eram classificados como transição/estabilidade. Para evitar uma leitura excessivamente neutra, foi criado o cenário "Deterioração do risco".

In [43]:
def classificar_cenario_refinado(
    movimento_concessoes,
    movimento_inadimplencia,
    valor_concessoes,
    valor_inadimplencia
):
    if pd.isna(valor_concessoes) or pd.isna(valor_inadimplencia):
        return "Sem histórico suficiente"

    if (
        movimento_concessoes == "Alta"
        and movimento_inadimplencia == "Alta"
    ):
        return "Crescimento com alerta"

    elif (
        movimento_concessoes == "Alta"
        and movimento_inadimplencia in ["Estável", "Queda"]
    ):
        return "Expansão sem deterioração observada"

    elif (
        movimento_concessoes == "Estável"
        and movimento_inadimplencia == "Alta"
    ):
        return "Deterioração do risco"

    elif (
        movimento_concessoes == "Queda"
        and movimento_inadimplencia == "Alta"
    ):
        return "Aperto com deterioração"

    elif (
        movimento_concessoes in ["Estável", "Queda"]
        and movimento_inadimplencia == "Queda"
    ):
        return "Recuperação do risco"

    else:
        return "Transição/estabilidade"

In [44]:
base_metricas["cenario_nominal_refinado"] = base_metricas.apply(
    lambda linha: classificar_cenario_refinado(
        linha["movimento_concessoes"],
        linha["movimento_inadimplencia"],
        linha["concessoes_var_12m_pct"],
        linha["inadimplencia_var_12m_pp"]
    ),
    axis=1
)

In [45]:
base_metricas["cenario_real_refinado"] = base_metricas.apply(
    lambda linha: classificar_cenario_refinado(
        linha["movimento_concessoes_real"],
        linha["movimento_inadimplencia"],
        linha["concessoes_reais_var_12m_pct"],
        linha["inadimplencia_var_12m_pp"]
    ),
    axis=1
)

In [46]:
revisao_refinada = base_metricas[
    [
        "data",
        "concessoes_reais_var_12m_pct",
        "inadimplencia_var_12m_pp",
        "movimento_concessoes_real",
        "movimento_inadimplencia",
        "cenario_real_refinado"
    ]
].tail(12).copy()

colunas_numericas = [
    "concessoes_reais_var_12m_pct",
    "inadimplencia_var_12m_pp"
]

revisao_refinada[colunas_numericas] = (
    revisao_refinada[colunas_numericas].round(2)
)

display(revisao_refinada)

,data,concessoes_reais_var_12m_pct,inadimplencia_var_12m_pp,movimento_concessoes_real,movimento_inadimplencia,cenario_real_refinado
166,2025-01-01,24.72,0.46,Alta,Estável,Expansão sem deterioração observada
167,2025-02-01,22.24,0.88,Alta,Alta,Crescimento com alerta
168,2025-03-01,16.02,0.88,Alta,Alta,Crescimento com alerta
169,2025-04-01,11.04,1.19,Alta,Alta,Crescimento com alerta
170,2025-05-01,9.03,1.38,Alta,Alta,Crescimento com alerta
171,2025-06-01,10.05,1.77,Alta,Alta,Crescimento com alerta
172,2025-07-01,10.68,2.07,Alta,Alta,Crescimento com alerta
173,2025-08-01,8.93,2.63,Alta,Alta,Crescimento com alerta
174,2025-09-01,8.72,2.29,Alta,Alta,Crescimento com alerta
175,2025-10-01,9.53,2.51,Alta,Alta,Crescimento com alerta


In [47]:
base_metricas["cenario_principal"] = (
    base_metricas["cenario_real_refinado"]
)

ordem_cenarios_refinada = [
    "Expansão sem deterioração observada",
    "Crescimento com alerta",
    "Deterioração do risco",
    "Aperto com deterioração",
    "Recuperação do risco",
    "Transição/estabilidade",
    "Sem histórico suficiente"
]

resumo_principal = (
    base_metricas["cenario_principal"]
    .value_counts()
    .reindex(ordem_cenarios_refinada)
    .fillna(0)
    .astype(int)
    .rename("meses")
    .to_frame()
)

resumo_principal["participacao_pct"] = (
    resumo_principal["meses"]
    / len(base_metricas)
    * 100
).round(2)

display(resumo_principal)

,meses,participacao_pct
cenario_principal,,
Expansão sem deterioração observada,50,28.09
Crescimento com alerta,22,12.36
Deterioração do risco,9,5.06
Aperto com deterioração,29,16.29
Recuperação do risco,35,19.66
Transição/estabilidade,19,10.67
Sem histórico suficiente,14,7.87


In [48]:
cores_cenarios_refinados = {
    "Expansão sem deterioração observada": "#2E8B57",
    "Crescimento com alerta": "#F4A261",
    "Deterioração do risco": "#7B2CBF",
    "Aperto com deterioração": "#D62828",
    "Recuperação do risco": "#2C7FB8",
    "Transição/estabilidade": "#A0A0A0",
    "Sem histórico suficiente": "#D9D9D9"
}

dados_matriz_principal = base_metricas.dropna(
    subset=[
        "concessoes_reais_var_12m_pct",
        "inadimplencia_var_12m_pp"
    ]
).copy()

fig_matriz_principal = px.scatter(
    dados_matriz_principal,
    x="concessoes_reais_var_12m_pct",
    y="inadimplencia_var_12m_pp",
    color="cenario_principal",
    color_discrete_map=cores_cenarios_refinados,
    category_orders={
        "cenario_principal": ordem_cenarios_refinada
    },
    hover_name="cenario_principal",
    hover_data={
        "data": "|%b/%Y",
        "concessoes_reais_var_12m_pct": ":.2f",
        "inadimplencia_var_12m_pp": ":.2f",
        "cenario_principal": False
    },
    labels={
        "concessoes_reais_var_12m_pct":
            "Crescimento real das concessões em 12 meses (%)",
        "inadimplencia_var_12m_pp":
            "Variação da inadimplência em 12 meses (p.p.)",
        "cenario_principal": "Cenário"
    },
    title="Matriz de cenários reais do crédito pessoal não consignado"
)

# Faixas consideradas estáveis
fig_matriz_principal.add_vrect(
    x0=-limite_concessoes_real,
    x1=limite_concessoes_real,
    fillcolor="gray",
    opacity=0.10,
    line_width=0
)

fig_matriz_principal.add_hrect(
    y0=-limite_inadimplencia_teste,
    y1=limite_inadimplencia_teste,
    fillcolor="gray",
    opacity=0.10,
    line_width=0
)

# Linhas centrais
fig_matriz_principal.add_vline(
    x=0,
    line_dash="dash",
    line_color="gray"
)

fig_matriz_principal.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray"
)

# Destacar dezembro de 2025
ultimo_mes = dados_matriz_principal.iloc[-1]

fig_matriz_principal.add_annotation(
    x=ultimo_mes["concessoes_reais_var_12m_pct"],
    y=ultimo_mes["inadimplencia_var_12m_pp"],
    text="Dez/2025",
    showarrow=True,
    arrowhead=2,
    ax=50,
    ay=-40
)

fig_matriz_principal.update_traces(
    marker=dict(size=8, opacity=0.85)
)

fig_matriz_principal.update_layout(
    template="plotly_white",
    height=650,
    legend_title="Cenário"
)

fig_matriz_principal.show()

In [49]:
cenarios_validos = base_metricas[
    base_metricas["cenario_principal"]
    != "Sem histórico suficiente"
].copy()

ordem_cenarios_validos = [
    "Expansão sem deterioração observada",
    "Crescimento com alerta",
    "Deterioração do risco",
    "Aperto com deterioração",
    "Recuperação do risco",
    "Transição/estabilidade"
]

resumo_principal_validos = (
    cenarios_validos["cenario_principal"]
    .value_counts()
    .reindex(ordem_cenarios_validos)
    .fillna(0)
    .astype(int)
    .rename("meses")
    .to_frame()
)

resumo_principal_validos["participacao_pct"] = (
    resumo_principal_validos["meses"]
    / len(cenarios_validos)
    * 100
).round(2)

display(resumo_principal_validos)

print(
    "Meses sem histórico suficiente:",
    (base_metricas["cenario_principal"]
     == "Sem histórico suficiente").sum()
)

,meses,participacao_pct
cenario_principal,,
Expansão sem deterioração observada,50,30.49
Crescimento com alerta,22,13.41
Deterioração do risco,9,5.49
Aperto com deterioração,29,17.68
Recuperação do risco,35,21.34
Transição/estabilidade,19,11.59


Meses sem histórico suficiente: 14


In [50]:
fig_linha_tempo_principal = px.scatter(
    cenarios_validos,
    x="data",
    y="cenario_principal",
    color="cenario_principal",
    color_discrete_map=cores_cenarios_refinados,
    category_orders={
        "cenario_principal": ordem_cenarios_validos
    },
    hover_name="cenario_principal",
    hover_data={
        "data": "|%b/%Y",
        "concessoes_reais_var_12m_pct": ":.2f",
        "inadimplencia_var_12m_pp": ":.2f",
        "cenario_principal": False
    },
    labels={
        "data": "Período",
        "cenario_principal": "Cenário",
        "concessoes_reais_var_12m_pct":
            "Concessões reais em 12 meses (%)",
        "inadimplencia_var_12m_pp":
            "Inadimplência em 12 meses (p.p.)"
    },
    title="Linha do tempo dos cenários reais de crédito"
)

fig_linha_tempo_principal.update_traces(
    marker=dict(size=9, opacity=0.90)
)

fig_linha_tempo_principal.add_vrect(
    x0="2020-03-01",
    x1="2021-12-01",
    fillcolor="#F4A261",
    opacity=0.10,
    line_width=0,
    annotation_text="Período da pandemia",
    annotation_position="top left"
)

fig_linha_tempo_principal.update_layout(
    template="plotly_white",
    height=600,
    legend_title="Cenário"
)

fig_linha_tempo_principal.show()

In [51]:
# Salvar gráficos definitivos
fig_matriz_principal.write_html(
    pasta_graficos / "matriz_cenarios_reais_refinada.html",
    include_plotlyjs="cdn"
)

fig_linha_tempo_principal.write_html(
    pasta_graficos / "linha_tempo_cenarios_reais_refinada.html",
    include_plotlyjs="cdn"
)

# Atualizar o CSV com a classificação definitiva
base_metricas.to_csv(
    pasta_dados_processados / "base_metricas_bcb.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

print("Cenários e gráficos definitivos salvos!")
print("Formato atualizado:", base_metricas.shape)

Cenários e gráficos definitivos salvos!
Formato atualizado: (178, 29)


## Síntese executiva

A tabela abaixo resume o último período disponível e reúne os principais indicadores usados na tomada de decisão.

In [52]:
ultimo_mes = base_metricas.iloc[-1]


def numero_br(valor, casas=2):
    return (
        f"{valor:,.{casas}f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


resumo_ultimo_mes = pd.DataFrame(
    {
        "Indicador": [
            "Período",
            "Concessões mensais",
            "Crescimento nominal das concessões",
            "Crescimento real das concessões",
            "Crescimento nominal do saldo",
            "Crescimento real do saldo",
            "Inadimplência não consignado",
            "Inadimplência consignado",
            "Diferencial de inadimplência",
            "Variação anual da inadimplência",
            "Taxa do crédito não consignado",
            "Selic acumulada no mês",
            "Prêmio bruto sobre a Selic",
            "Cenário principal"
        ],
        "Resultado": [
            ultimo_mes["data"].strftime("%m/%Y"),
            "R$ "
            + numero_br(ultimo_mes["concessoes_milhoes"], 0)
            + " milhões",
            numero_br(ultimo_mes["concessoes_var_12m_pct"]) + "%",
            numero_br(
                ultimo_mes["concessoes_reais_var_12m_pct"]
            ) + "%",
            numero_br(ultimo_mes["saldo_var_12m_pct"]) + "%",
            numero_br(ultimo_mes["saldo_real_var_12m_pct"]) + "%",
            numero_br(
                ultimo_mes["inadimplencia_nao_consignado_pct"]
            ) + "%",
            numero_br(
                ultimo_mes["inadimplencia_consignado_pct"]
            ) + "%",
            numero_br(
                ultimo_mes["diferencial_inadimplencia_pp"]
            ) + " p.p.",
            numero_br(
                ultimo_mes["inadimplencia_var_12m_pp"]
            ) + " p.p.",
            numero_br(
                ultimo_mes["juros_nao_consignado_pct_mes"]
            ) + "% ao mês",
            numero_br(ultimo_mes["selic_pct_mes"]) + "%",
            numero_br(
                ultimo_mes["premio_sobre_selic_pp"]
            ) + " p.p.",
            ultimo_mes["cenario_principal"]
        ]
    }
)

display(resumo_ultimo_mes)

,Indicador,Resultado
0,Período,12/2025
1,Concessões mensais,R$ 21.753 milhões
2,Crescimento nominal das concessões,"8,22%"
3,Crescimento real das concessões,"3,59%"
4,Crescimento nominal do saldo,"18,93%"
5,Crescimento real do saldo,"14,07%"
6,Inadimplência não consignado,"9,16%"
7,Inadimplência consignado,"2,81%"
8,Diferencial de inadimplência,"6,35 p.p."
9,Variação anual da inadimplência,"2,70 p.p."


### Conclusão

Em dezembro de 2025, o crédito pessoal não consignado apresentou sinais de desaceleração acompanhados por deterioração do risco.

**Demanda:** as concessões cresceram 8,22% em valores nominais, mas apenas 3,59% depois do ajuste pelo IPCA. Isso demonstra que parte relevante do crescimento observado correspondia à inflação.

**Carteira:** o saldo apresentou crescimento real de 14,07%, significativamente superior ao crescimento real das novas concessões. O estoque de crédito permaneceu em expansão, mesmo com a desaceleração do fluxo de novas operações.

**Risco:** a inadimplência do crédito não consignado atingiu 9,16%, com aumento de 2,70 pontos percentuais em doze meses. O indicador ficou 6,35 pontos percentuais acima da inadimplência do consignado.

**Preço:** a taxa média do crédito não consignado foi de 6,65% ao mês, frente a uma Selic mensal de 1,22%. A diferença de 5,43 pontos percentuais representa um prêmio bruto sobre a Selic, não o spread bancário oficial.

**Cenário:** a combinação entre concessões reais relativamente estáveis e inadimplência crescente levou à classificação de **Deterioração do risco**.

Do ponto de vista de negócio, o resultado sugere maior atenção à qualidade das novas concessões, aos critérios de crédito, à cobrança e ao preço ajustado ao risco. A análise não implica uma recomendação automática de interrupção do crédito.

Como os dados são agregados, as conclusões representam uma leitura de mercado e portfólio. Elas não permitem avaliar clientes individualmente nem estabelecer relações causais.

### 12.2 Teste de robustez ampliado

A classificação principal usa o percentil neutro de 20% e a média móvel de três meses. Para avaliar a estabilidade da leitura, serão comparados os percentis 10%, 20% e 30% e as médias móveis de três e seis meses.

O teste mede a concordância de cada combinação com a especificação principal. Ele não transforma a regra em modelo preditivo; apenas mostra quanto a classificação depende das escolhas metodológicas.

In [53]:
from src.analysis import testar_robustez

teste_robustez = testar_robustez(
    base_analitica,
    percentis=(0.10, 0.20, 0.30),
    janelas=(3, 6),
)

display(teste_robustez)

,media_movel_meses,percentil_neutro,concordancia_com_referencia_pct,cenarios_distintos,observacoes
0,3,10,85.365854,6,164
1,3,20,100.000000,6,164
2,3,30,91.463415,6,164
3,6,10,81.987578,6,161
4,6,20,90.683230,6,161
5,6,30,85.714286,6,161


### 13.5 Associação descritiva com inadimplência futura

Concessões são um fluxo de novas operações, enquanto a inadimplência representa o risco observado no estoque da carteira. Portanto, movimentos de originação podem se associar à inadimplência apenas depois de alguns meses.

A análise abaixo compara a variação em 12 meses das concessões reais atuais com a variação da inadimplência 3, 6, 9 e 12 meses à frente. Os coeficientes são associações descritivas de Pearson: **não demonstram causalidade**.

In [54]:
from src.analysis import analisar_defasagens

analise_defasagens = analisar_defasagens(
    base_metricas,
    defasagens=(3, 6, 9, 12),
)

display(analise_defasagens)

fig_defasagens = px.bar(
    analise_defasagens,
    x="defasagem_meses",
    y="correlacao_pearson",
    text_auto=".2f",
    labels={
        "defasagem_meses": "Inadimplência futura (meses)",
        "correlacao_pearson": "Correlação de Pearson",
    },
    title="Associação descritiva: concessões atuais - inadimplência futura",
)
fig_defasagens.add_hline(y=0, line_color="gray")
fig_defasagens.show()

,defasagem_meses,correlacao_pearson,observacoes
0,3,-0.027432,161
1,6,0.175698,158
2,9,0.339298,155
3,12,0.384767,152


### 13.6 Cobertura, qualidade e fórmulas

A base consolidada cobre março de 2011 a dezembro de 2025. A validação documenta quantidade de meses, datas duplicadas e ausências. Valores ausentes no início de indicadores derivados são esperados quando a fórmula exige média móvel ou comparação com o mesmo mês do ano anterior.

As variações de volume e risco são priorizadas em 12 meses para reduzir a influência da sazonalidade mensal.

| Indicador | Fórmula |
|---|---|
| Concessões reais | concessões nominais �- índice IPCA do último mês ÷ índice IPCA do mês |
| Crescimento real das concessões | variação em 12 meses da média móvel das concessões reais |
| Crescimento real do saldo | variação em 12 meses do saldo corrigido pelo IPCA |
| Variação da inadimplência | inadimplência atual �^' inadimplência do mesmo mês do ano anterior |
| Diferencial de inadimplência | inadimplência não consignado �^' inadimplência consignado |
| Prêmio bruto sobre a Selic | taxa mensal do crédito pessoal �^' Selic mensal |

O prêmio bruto sobre a Selic é apenas uma diferença de taxas e não incorpora todos os componentes do spread bancário.

In [55]:
from src.analysis import resumir_qualidade

resumo_qualidade = resumir_qualidade(base_analitica)
display(resumo_qualidade)

teste_robustez.to_csv(
    pasta_dados_processados / "teste_robustez.csv",
    index=False,
    encoding="utf-8-sig",
)
analise_defasagens.to_csv(
    pasta_dados_processados / "analise_defasagens.csv",
    index=False,
    encoding="utf-8-sig",
)

,inicio,fim,meses,datas_duplicadas,ausencias_total,meses_ausentes,lista_meses_ausentes
0,2011-03-01,2025-12-01,178,0,0,0,Nenhum


### 13.7 Classificação temporal e risco de look-ahead

A classificação principal é retrospectiva: os percentis são calculados com a amostra completa e, por isso, incorporam informação posterior ao mês classificado. Para simular monitoramento em tempo real, a alternativa abaixo usa uma janela expansiva e estima cada limite somente com observações anteriores, exigindo ao menos 24 observações válidas. As duas leituras têm finalidades distintas; nenhuma delas identifica causalidade.


In [56]:
from src.analysis import comparar_classificacao_temporal

comparacao_temporal, resumo_temporal = comparar_classificacao_temporal(base_analitica)
display(resumo_temporal)
display(comparacao_temporal.tail(12))
comparacao_temporal.to_csv(pasta_dados_processados / 'comparacao_classificacao_temporal.csv', index=False, encoding='utf-8-sig')
resumo_temporal.to_csv(pasta_dados_processados / 'resumo_classificacao_temporal.csv', index=False, encoding='utf-8-sig')


,observacoes_comparaveis,concordancia_pct,divergencias
0,140,95.714286,6


,data,cenario_retrospectivo,cenario_sem_lookahead,classificacoes_concordam
166,2025-01-01,Expansão sem deterioração observada,Expansão sem deterioração observada,True
167,2025-02-01,Crescimento com alerta,Crescimento com alerta,True
168,2025-03-01,Crescimento com alerta,Crescimento com alerta,True
169,2025-04-01,Crescimento com alerta,Crescimento com alerta,True
170,2025-05-01,Crescimento com alerta,Crescimento com alerta,True
171,2025-06-01,Crescimento com alerta,Crescimento com alerta,True
172,2025-07-01,Crescimento com alerta,Crescimento com alerta,True
173,2025-08-01,Crescimento com alerta,Crescimento com alerta,True
174,2025-09-01,Crescimento com alerta,Crescimento com alerta,True
175,2025-10-01,Crescimento com alerta,Crescimento com alerta,True
